<a href="https://colab.research.google.com/github/Fida-Ukwishaka/Lab-4-LLM-decision-support/blob/main/lab_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [24]:
# Part 0: Repository and API-key setup

In [25]:
# API-key setup — DO NOT hard-code your key in this cell.

import os

# --- Local (with a .env file) ---
# from dotenv import load_dotenv
# load_dotenv()
# API_KEY = os.environ["GROQ_API_KEY"]

# --- Google Colab (Secrets panel) ---
# from google.colab import userdata
from google.colab import userdata
# API_KEY = userdata.get("GROQ_API_KEY")
# TODO: set API_KEY using ONE of the methods above.
API_KEY = userdata.get("GROQ_API_KEY")
# OpenAI-compatible client (works for Groq and OpenAI; Gemini users see their docs):
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",   # remove this line if using OpenAI itself
)
MODEL = "llama-3.3-70b-versatile"                # or your provider's model name

print("Client ready.")


Client ready.


In [26]:
# Section 1 - Talking to an LLM Programmatically

In [27]:
# Part 1.1 - Your first API call

In [28]:
# TODO: Write a helper function you will reuse for the WHOLE lab:
#
# def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
def ask_llm(user_prompt, system_prompt="You are a helpful assistant.", temperature =0.7, max_tokens=500):
#             temperature=0.7, max_tokens=500):
# TODO: Call it once with a simple question and print the answer.
#     response = client.chat.completions.create(
    response = client.chat.completions.create(
#         model=MODEL,
#         messages=[
#             {"role": "system", "content": system_prompt},
#             {"role": "user",   "content": user_prompt},
#         ],
#         temperature=temperature,
#         max_tokens=max_tokens,
#     )
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
#     return response.choices[0].message.content
    return response.choices[0].message.content, response.usage
#
# TODO: Call it once with a simple question and print the answer.
answer, usage = ask_llm("What is the capital of France?")
print("Answer: ",answer)
# TODO: Print response.usage as well — how many tokens did your call consume?
print("\nToken usage: ",usage)


Answer:  The capital of France is Paris.

Token usage:  CompletionUsage(completion_tokens=8, prompt_tokens=48, total_tokens=56, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.010253129, prompt_time=0.003199945, completion_time=0.010841835, total_time=0.01404178)


Student Reasoning

1. System gives the model its overall instructions or behaviour. It tells the AI how it should behave. On the other hand,
User contains the actual question or request from the person using the AI.

2. A token is a small piece of text that the language model processes. It can be a word, part of a word, or punctuation.

 API providers charge per token because different requests use different amounts of text and computational resources,  so token-based pricing more accurately reflects the amount of work required.



In [29]:
# Part 1.2 - Temperature: the randomness dial

In [30]:
# TODO: Ask the SAME question 5 times at temperature=0.0 and 5 times at temperature=1.2.
#   A good test question: "Suggest a name for a savings product for market traders in Accra."
# TODO: Print all 10 answers, grouped by temperature.
question = "Suggest a name for a savings product for market traders in Accra."

# Temperature = 0
answers_temp_0 = []

for i in range(5):
    answer, usage = ask_llm(question, temperature=0)
    answers_temp_0.append(answer)

# Temperature = 1.2
answers_temp_1_2 = []

for i in range(5):
    answer, usage = ask_llm(question, temperature=1.2)
    answers_temp_1_2.append(answer)

# Print all answers
print(" TEMPERATURE = 0 ")

for i, answer in enumerate(answers_temp_0, 1):
    print(f"\nAnswer {i}:")
    print(answer)

print("\n TEMPERATURE = 1.2 ")

for i, answer in enumerate(answers_temp_1_2, 1):
    print(f"\nAnswer {i}:")
    print(answer)

 TEMPERATURE = 0 

Answer 1:
Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **Trader's Treasure**: This name emphasizes the idea of saving and accumulating wealth.
3. **Sika Souce**: "Sika" means "money" in the Akan language, which is widely spoken in Ghana. "Souce" is a play on the word "source," implying a reliable and trustworthy savings product.
4. **Market Booster**: This name suggests that the savings product can help market traders boost their businesses and achieve their financial goals.
5. **Adanfo Save**: "Adanfo" means "friends" or "partners" in the Akan language, implying a sense of community and mutual support among market traders.
6. **Kokroko Savings**: "Kokroko" is a Ghanaian term that means "honest" or "transparent," which could help build trust with potential customers.
7. **Traders' Fund**: This name is straightforward and 

Student Reasoning

At temperature 0, the responses were more consistent and similar across the five attempts. At temperature 1.2, the responses were more varied ans creative.
For the loan decison-support system, I would use low temperature because loan-related recommendations should be consistent, predictable, and reliable rather than highlty creative or random. The system should not give significantly different recomendations simply because the model generated a different response.

In [31]:
# Section 2 - The Dataset: Loan Application Letters

In [32]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.


In [33]:
# Section 3 - Prompt Engineering for the Decision Support System

In [34]:
# 3.2 - Component 1: Summarization

In [35]:

# TODO: Write SUMMARY_PROMPT_V1 — your first, naive attempt (e.g. just "Summarize this:").
#   Run it on L002 and L006. Read the output critically.
SUMMARY_PROMPT_V1 = "Summarize this:"
SUMMARY_PROMPT_V1_L002, usage = ask_llm(f"{SUMMARY_PROMPT_V1}\n\n{LETTERS['L002']}")
SUMMARY_PROMPT_V1_L006, usage = ask_llm(f"{SUMMARY_PROMPT_V1}\n\n{LETTERS['L006']}")
print(f"\nSUMMARY_PROMPT_V1_L002:\n{SUMMARY_PROMPT_V1_L002}\n")
print(f"\nSUMMARY_PROMPT_V1_L006:\n{SUMMARY_PROMPT_V1_L006}\n")
# TODO: Now write SUMMARY_PROMPT_V2 as a proper template with:
#   - a system prompt giving the LLM a ROLE (e.g. "You are an assistant to a microfinance
#     loan officer...") and constraints (factual, neutral, no invented details, 3-4 sentences)
#   - a user prompt template like: f"Summarize this loan application:\n\n{letter_text}"
#   Run V2 on the same two letters at temperature=0.
SUMMARY_PROMPT_V2 = """You are an assistant to a microfinance loan officer.
Summarize the loan application factually and neutrally.
Do not invent or assume any details that are not stated in the application.
Keep the summary to 3-4 sentences."""
user_prompt_L002 = f"Summarize this loan application:\n\n{LETTERS['L002']}"
user_prompt_L006 = f"Summarize this loan application:\n\n{LETTERS['L006']}"
SUMMARY_PROMPT_V2_L002, usage = ask_llm(SUMMARY_PROMPT_V2, user_prompt_L002, temperature=0)
SUMMARY_PROMPT_V2_L006, usage = ask_llm(SUMMARY_PROMPT_V2, user_prompt_L006, temperature=0)
print(f"\nSUMMARY_PROMPT_V2_L002:\n{SUMMARY_PROMPT_V2_L002}\n")
print(f"\nSUMMARY_PROMPT_V2_L006:\n{SUMMARY_PROMPT_V2_L006}\n")
# TODO: Compare V1 vs V2 outputs side by side. Keep both prompt versions in this notebook.
print("========== L002 ==========")

print("\n--- V1: Naive Prompt ---")
print(SUMMARY_PROMPT_V1_L002)

print("\n--- V2: Structured Prompt ---")
print(SUMMARY_PROMPT_V2_L002)


print("\n\n========== L006 ==========")

print("\n--- V1: Naive Prompt ---")
print(SUMMARY_PROMPT_V1_L006)

print("\n--- V2: Structured Prompt ---")
print(SUMMARY_PROMPT_V2_L006)


SUMMARY_PROMPT_V1_L002:
Kwame Boateng, a commercial driver from Kumasi, is seeking a loan of GHS 25,000 to repair his vehicle's engine and pay off personal debts. He's experiencing a slow period in business, but expects it to improve after the festive season and is willing to repay the loan when he can. He doesn't have collateral to offer at the moment.


SUMMARY_PROMPT_V1_L006:
Kofi, a 22-year-old, is requesting GHS 50,000 to start three businesses: a car washing service, a provision shop, and a phone import business from Dubai. He has no experience or collateral, but claims to be "business-minded" and trustworthy, promising to repay the loan within a year when his businesses are successful.


SUMMARY_PROMPT_V2_L002:
Kwame Boateng, a commercial driver in Kumasi, has applied for a loan of GHS 25,000. He states that the loan is needed to repair his trotro engine and settle personal debts. Mr. Boateng mentions that his business has been slow, but he expects it to improve after the festi

Student Reasoning

1. V1 was more detailed and included information that was not necessarily important to the loan summary. Fro eaxample, for L002, V1 says Kame is "Urgently seeking GHS 25000" and mentions that he wants to pay off personal debts. V2 is more focused, stating that he "has applied for a loan of GHS 25000" and explaining the stated purpose of the loan. For L006, V1 includes Kofi's age, while V2 focuses more directly on the loan amount and the three intended businesses. The structured prompt therefore makes the summaries more focused, neutral, and appropriate for a loan officer.

2. "No invented" details is essential because the summary may be used to support a real loan decision. If the model adds information that was not provided by the applicant, the loan officer could mistake it for a fact and make an unfair or incorrect decision. An LLM generating unsupported or false information is called a halluciantion.

In [36]:
# Part 3.2 - Component 2: Structured extraction (JSON)

In [37]:
# TODO: Write EXTRACT_PROMPT — a template that instructs the model to return ONLY a JSON
#   object with EXACTLY these keys:
#     applicant_name (string), amount_ghs (number), purpose (string),
#     monthly_profit_ghs (number or null), has_collateral_or_guarantor (boolean),
#     repayment_months (number or null)
#   Techniques to use:
#     - explicit schema in the prompt
#     - ONE worked example (few-shot) using a letter you write yourself (not from LETTERS!)
#     - "If a field is not stated in the letter, use null. Do not guess."
#     - temperature=0
EXTRACT_PROMPT = """
You are an information extraction assistant for a microfinance loan decision-support system.

Extract information ONLY from the letter provided.

Return ONLY a valid JSON object with EXACTLY these keys:

{
  "applicant_name": "string",
  "amount_ghs": number,
  "purpose": "string",
  "monthly_profit_ghs": number or null,
  "has_collateral_or_guarantor": boolean,
  "repayment_months": number or null
}

Rules:
- If a field is not stated in the letter, use null.
- Do not guess or infer missing information.
- amount_ghs must be a number.
- monthly_profit_ghs must be a number or null.
- repayment_months must be a number or null.
- has_collateral_or_guarantor must be true if collateral or a guarantor is explicitly mentioned, otherwise false.
- Return ONLY the JSON object.
- Do not include explanations.
- Do not include markdown or ```json fences.
- Do not add any extra keys.

Example:

Letter:
"My name is Ama Mensimah. I am requesting GHS 10,000 to buy equipment for my bakery.
The bakery makes GHS 1,200 profit per month. My brother will guarantee the loan.
I will repay the loan over 10 months."

Correct output:
{
  "applicant_name": "Ama Mensimah",
  "amount_ghs": 10000,
  "purpose": "buy equipment for bakery",
  "monthly_profit_ghs": 1200,
  "has_collateral_or_guarantor": true,
  "repayment_months": 10
}

Now extract the information from this letter:

{letter_text}
"""
# TODO: Write extract_fields(letter_text) that calls the LLM, strips any ```json fences,
#   json.loads() the result, and returns a dict. Handle parse failures gracefully
#   (return None and print a warning).
import json

def extract_fields(letter_text, temperature = 0):
    prompt = EXTRACT_PROMPT.replace("{letter_text}", letter_text)

    try:
        response, usage = ask_llm(
            prompt,
            temperature=0
        )

        response = response.strip()

        # Remove markdown JSON fences if present
        if response.startswith("```json"):
            response = response[7:].strip()
        elif response.startswith("```"):
            response = response[3:].strip()

        if response.endswith("```"):
            response = response[:-3].strip()

        result = json.loads(response)

        return result

    except json.JSONDecodeError:
        print("Warning: Could not parse the LLM response as JSON.")
        return None

    except Exception as e:
        print(f"Warning: Extraction failed: {e}")
        return None
# TODO: Run it on ALL SIX letters; collect results into a pandas DataFrame (one row per
#   letter) and display it.
import pandas as pd

results = []

for letter_id, letter_text in LETTERS.items():
    result = extract_fields(letter_text)

    if result is not None:
        result["letter_id"] = letter_id
        results.append(result)

df = pd.DataFrame(results)

display(df)

,applicant_name,amount_ghs,purpose,monthly_profit_ghs,has_collateral_or_guarantor,repayment_months,letter_id
0,Akosua Mensah,8000,buy a deep freezer and expand into frozen foods,900.0,True,20.0,L001
1,Kwame Boateng,25000,repair trotro engine and settle personal debts,NaN,False,NaN,L002
2,Efua Darko,15000,purchase two industrial sewing machines and fa...,2800.0,True,15.0,L003
3,Yaw Owusu,12000,for feed and 500 new layers for poultry farm,1500.0,True,18.0,L004
4,Adenta Women's Weaving Cooperative,30000,buy a bulk order of yarn,NaN,True,16.0,L005
5,Kofi,50000,"start a car washing business, a provision shop...",NaN,False,12.0,L006


Student Reasoning

1. Because using one of the six letters as the example could leak the answer to the model. The model might remember or copy information from the example instead of actually extracting the information from each new letter. Using a separate example tests whether the model can genuinely perform the extraction task.

2. Without this instruction, the model may make up or infer missing information. For example, if a letter does not mention the repayment period, the model might assume a reasonable number of months instead of leaving it blank. Using null makes it clear that the information was not provided.

3. temperature=0 makes the model's responses more consistent and predictable. This is useful for extraction because we want the same information to be identified accurately and formatted the same way every time.
For creative tasks, however, some randomness is useful because it can produce different, more original ideas and wording. Therefore, a higher temperature can be better for creative writing or brainstorming.

In [38]:
# Part 3.3 - Compnent 3: The decision-support brief

In [39]:
# TODO: Write BRIEF_PROMPT — it receives the letter AND your extracted JSON, and must output:
#     1. Strengths (bullet points, grounded in the letter)
#     2. Risks / red flags (bullet points)
#     3. Missing information the officer should request
#     4. Suggested next step (e.g. "invite for interview", "request documents",
#        "flag for senior review") — NOT "approve" or "reject".
#   Give the model an explicit instruction that final decisions are made by humans.
BRIEF_PROMPT = """
You are assisting a microfinance loan officer.

Using the original loan application letter and the extracted JSON, produce a short decision-support brief.

Your response MUST contain exactly these four sections:

1. Strengths
- Use bullet points.
- Every strength must be grounded in information explicitly stated in the letter.

2. Risks / Red Flags
- Use bullet points.
- Identify concerns or risk factors based only on the letter and extracted information.
- Do not invent facts.

3. Missing Information
- Use bullet points.
- List information that the loan officer should request if it is missing or unclear.

4. Suggested Next Step
- Give a practical next step for the loan officer.
- Examples include "invite for interview", "request documents", or "flag for senior review".
- Do NOT say "approve" or "reject".

IMPORTANT:
- The final lending decision is made by a human loan officer.
- You are providing decision support only, not making the final decision.
- Do not invent or assume information that is not present.
- Keep the brief concise and clear.

ORIGINAL LETTER:
{letter_text}

EXTRACTED JSON:
{extracted_json}
"""
# TODO: Generate briefs for ALL SIX letters. Print the briefs for L001, L002, and L006 —
#   three very different applications.
briefs = {}

for letter_id, letter_text in LETTERS.items():
    extracted = extract_fields(letter_text)

    if extracted is not None:
        extracted_json = json.dumps(extracted, indent=2)

        prompt = BRIEF_PROMPT.replace("{letter_text}", letter_text)
        prompt = prompt.replace("{extracted_json}", extracted_json)

        response, usage = ask_llm(
            prompt,
            temperature=0
        )

        briefs[letter_id] = response

for letter_id in ["L001", "L002", "L006"]:
    print(f"\n{'=' * 60}")
    print(f"BRIEF FOR {letter_id}")
    print(f"{'=' * 60}")
    print(briefs[letter_id])


BRIEF FOR L001
## 1. Strengths
* The applicant, Akosua Mensah, has 12 years of experience selling provisions at Makola Market.
* She has a stable monthly profit of GHS 900 from her current stall.
* Akosua has saved GHS 2,500 with the susu scheme over two years without missing a contribution, demonstrating her ability to manage savings.
* She has a guarantor, her sister, who is a teacher, adding a level of security to the loan.
* Akosua has a clear plan for repayment, proposing to pay GHS 450 monthly over 20 months.

## 2. Risks / Red Flags
* The loan amount of GHS 8,000 is significant compared to her monthly profit, which might pose a risk if her expanded business does not generate enough additional income.
* There is no detailed information on the sister's financial stability or ability to act as a guarantor.
* The application does not provide specifics on how the deep freezer and expansion into frozen foods will increase her profits.

## 3. Missing Information
* Detailed financial p

Student Reasoning

1. Yes. For L003, the system should identify strengths such as strong business performance, ability to repay, and other positive evidence clearly stated in the letter. Its risks should be relatively limited, with any missing information highlighted for the officer.

For L006, the system should identify the weaknesses more clearly, such as limited evidence of repayment ability, missing financial information, lack of collateral/guarantor if applicable, or other concerns stated in the letter. Therefore, the briefs should show a clear difference between the stronger and weaker applications rather than treating them the same.

2. -Practical reason: The model may misunderstand information or make an incorrect judgment. The loan officer needs to review the evidence and make the final decision.
-Ethical reason: Loan approval affects a person's access to money and can significantly affect their life. Allowing an AI to automatically approve or reject applicants could lead to unfair or biased decisions without proper human oversight.

In short, The AI provides evidence and recommendations, while the human loan officer makes the final decision.


In [40]:
# Part 3.4 Commit your prompt templates

In [41]:
# Section 4 - Evaluation: Quality, Reliability, Appropriateness

In [42]:
# Part 4.1 - Extraction accuracy against gold labels

In [43]:
# TODO: For the three letters in GOLD, compare your extracted DataFrame to the gold values
#   field by field. Compute per-field accuracy across the three letters
#   (name matching can be case-insensitive; numbers must match exactly).
gold_ids = ["L001", "L003", "L006"]

fields = [
    "applicant_name",
    "amount_ghs",
    "purpose",
    "monthly_profit_ghs",
    "has_collateral_or_guarantor",
    "repayment_months"
]

rows = []

for field in fields:
    row = {}
    correct_count = 0

    for letter_id in gold_ids:
        extracted = df.loc[df["letter_id"] == letter_id, field].iloc[0]
        gold = GOLD[letter_id][field]

        if field == "applicant_name":
            correct = str(extracted).strip().lower() == str(gold).strip().lower()
        else:
            correct = extracted == gold

        row[letter_id] = int(correct)
        correct_count += int(correct)

    row["accuracy"] = correct_count / len(gold_ids)
    rows.append(row)

# TODO: Display a small table: rows = fields, columns = L001 / L003 / L006 / accuracy.
accuracy_df = pd.DataFrame(
    rows,
    index=fields
)

display(accuracy_df)

,L001,L003,L006,accuracy
applicant_name,1,1,1,1.000000
amount_ghs,1,1,1,1.000000
purpose,0,0,0,0.000000
monthly_profit_ghs,1,1,0,0.666667
has_collateral_or_guarantor,1,1,1,1.000000
repayment_months,1,1,1,1.000000


In [44]:
# Part 4.2 Reliability: is the system consistent?

In [45]:
# TODO: Run extract_fields() on letter L004 FIVE times at temperature=0 and FIVE times at
#   temperature=1.0.
results_by_temp = {}

for temperature in [0, 1.0]:
    runs = []

    for i in range(5):
        result = extract_fields(
            LETTERS["L004"],
            temperature=temperature
        )

        runs.append(result)

    results_by_temp[temperature] = runs

# TODO: For each temperature, report how many of the 5 runs produced (a) valid JSON and
#   (b) identical values across runs. A simple approach: json.dumps(result, sort_keys=True)
#   and count unique strings.
import json

for temperature, runs in results_by_temp.items():

    valid_json = sum(result is not None for result in runs)

    json_strings = [
        json.dumps(result, sort_keys=True)
        for result in runs
        if result is not None
    ]

    unique_values = len(set(json_strings))

    print(f"Temperature = {temperature}")
    print(f"Valid JSON: {valid_json}/5")
    print(f"Unique values: {unique_values}")
    print()

Temperature = 0
Valid JSON: 5/5
Unique values: 1

Temperature = 1.0
Valid JSON: 5/5
Unique values: 1



In [ ]:
# Part 4.3 - Hallucination probing

In [47]:
# TODO: Design TWO adversarial tests and run them:
#   Test 1 — Ask your summarizer a question about a detail that is NOT in a letter
#     (e.g. "What is the applicant's credit score?"). Does it admit the information is
#     absent, or does it invent one?
test1_prompt = f"""
Here is a loan application letter:

{LETTERS["L001"]}

What is the applicant's credit score?
"""

response1, usage = ask_llm(
    test1_prompt,
    temperature=0
)

print("TEST 1 OUTPUT:")
print(response1)
#   Test 2 — Feed your extractor an EMPTY or IRRELEVANT text (e.g. a weather report).
#     Does it return nulls, or does it fabricate an applicant?

weather_text = """
Today's weather forecast:
Accra will be sunny with temperatures reaching 31°C.
There may be light rain in the evening.
Winds will be moderate from the southwest.
"""

response2 = extract_fields(
    weather_text,
    temperature=0
)

print("TEST 2 OUTPUT:")
print(response2)
# TODO: Record the outputs verbatim below and label each PASS or FAIL.
print("TEST 1 OUTPUT:")
print(response1)
print("RESULT: FAIL")

print("\nTEST 2 OUTPUT:")
print(response2)
print("RESULT: PASS")

TEST 1 OUTPUT:
The applicant's credit score is not explicitly mentioned in the loan application letter. However, based on the information provided, we can infer that the applicant has a good credit history. 

The applicant has been consistently saving with the susu scheme for two years and has never missed a contribution, which suggests a high level of financial discipline and responsibility. Additionally, the applicant has a stable business with a monthly profit of GHS 900 and a clear plan for repaying the loan. 

Overall, while the credit score is not provided, the applicant's financial behavior and business stability suggest a positive credit profile.
TEST 2 OUTPUT:
{'applicant_name': None, 'amount_ghs': None, 'purpose': None, 'monthly_profit_ghs': None, 'has_collateral_or_guarantor': False, 'repayment_months': None}
TEST 1 OUTPUT:
The applicant's credit score is not explicitly mentioned in the loan application letter. However, based on the information provided, we can infer that th